In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
secondary_sales = pd.read_csv("../data/raw/secondary_sales.csv")
under_construction = pd.read_csv("../data/raw/under_construction.csv")

print(secondary_sales.isnull().sum())
print("\n")
print(under_construction.isnull().sum())

id                           0
date_listed                  0
locality                     0
region                       0
tier                         0
lat                          0
lon                          0
property_type                0
bedrooms                     0
carpet_area_sqft             0
built_up_area_sqft           0
carpet_area_m2               0
floor                        0
total_floors                 0
year_built                   0
facing                       0
furnishing                   0
parking                      0
balconies                    0
is_sra_redevelopment         0
builder                      0
builder_tier                 0
metro_station                0
metro_line                   0
metro_distance_min           0
metro_distance_type          0
to_bkc_km                    0
to_nariman_point_km          0
price_inr                    0
price_per_sqft_carpet_inr    0
price_usd                    0
home_loan_rate_at_listing    0
dtype: i

# Checking the Duplicates

In [3]:
print(secondary_sales.duplicated().sum())
print(under_construction.duplicated().sum())

0
0


In [5]:
secondary_sales['date_listed'] = pd.to_datetime(secondary_sales['date_listed'])

under_construction['date_listed'] = pd.to_datetime(
    under_construction['date_listed']
)



secondary_sales['listing_year'] = secondary_sales['date_listed'].dt.year
secondary_sales['listing_month'] = secondary_sales['date_listed'].dt.month

under_construction['listing_year'] = under_construction['date_listed'].dt.year
under_construction['listing_month'] = under_construction['date_listed'].dt.month

In [6]:
current_year = 2026

secondary_sales['property_age'] = (
    current_year - secondary_sales['year_built']
)

# PRICE PER SQFT Feature

In [7]:
secondary_sales['calculated_price_per_sqft'] = (
    secondary_sales['price_inr']
    /
    secondary_sales['carpet_area_sqft']
)

# LUXURY SCORE

In [8]:
secondary_sales['luxury_score'] = 0

secondary_sales.loc[
    secondary_sales['furnishing'] == 'fully_furnished',
    'luxury_score'
] += 2

secondary_sales.loc[
    secondary_sales['parking'] == 'covered',
    'luxury_score'
] += 2

secondary_sales.loc[
    secondary_sales['bedrooms'] >= 3,
    'luxury_score'
] += 2

secondary_sales.loc[
    secondary_sales['tier'] == 'luxury',
    'luxury_score'
] += 3

# CONNECTIVITY SCORE

In [10]:
secondary_sales['connectivity_score'] = (
    100
    - secondary_sales['metro_distance_min']
)

secondary_sales['connectivity_score'] = (
    secondary_sales['connectivity_score']
    .clip(lower=0)
)

# INVESTMENT SCORE (Unique feature)
Investment Score=0.4(Connectivity)+0.3(Luxury)+0.2(Price Growth Proxy)+0.1(Affordability)

In [11]:
secondary_sales['investment_score'] = (
    0.4 * secondary_sales['connectivity_score']
    +
    0.3 * secondary_sales['luxury_score']
    +
    0.2 * (
        secondary_sales['price_per_sqft_carpet_inr'] / 1000
    )
    +
    0.1 * (
        10000000 / secondary_sales['price_inr']
    )
)

# Investment Category

In [12]:
secondary_sales['investment_category'] = pd.cut(
    secondary_sales['investment_score'],
    bins=[0, 20, 50, 100],
    labels=['Low', 'Medium', 'High']
)

# SAVING CLEANED DATASET

In [13]:
secondary_sales.to_csv(
    "../data/processed/secondary_sales_cleaned.csv",
    index=False
)

under_construction.to_csv(
    "../data/processed/under_construction_cleaned.csv",
    index=False
)